# R06-H32 - cure timing must come from the evidence, not a constant

**Author**: Knowledge Graph Foundry autonomous build

Replays the wave-1 type-arrival stream (reconstructed from the live campaign graph in ingest order) through four candidate curing gates and compares each gate's cure document against the retrospective type-saturation band. Motivating incident: DEF-3 - the v1 plateau gate cured at document 4 of 481. Pre-registered in [kgf-redesign-experiments.md](../docs/experiments/kgf-redesign-experiments.md) under R06-H32.

Candidates: (a) v1 plateau (baseline, known premature), (b) chao1 composite convergence, (c) point-estimate missing mass behind a hardcoded 200-observation floor (the rejected first patch - the floor drives the result), (d) Good-Turing missing-mass upper confidence bound (scale-free; the minimum evidence mass emerges from the bound itself). Acceptance: (d) cures inside the saturation band with zero corpus-size input, and its verdict is invariant under stream truncation (100 / 200 / full).

In [1]:
# imports + replay data: per-document type observations in ingest order
import json
from collections import Counter
from pathlib import Path

import logging
logging.getLogger("neo4j").setLevel(logging.CRITICAL)
from neo4j import GraphDatabase

from knowledge_graph_foundry.ontology.metrics import StabilityMetrics
from knowledge_graph_foundry.ontology.curing import CuringDetector
from knowledge_graph_foundry.settings import CuringSettings

STRUCTURAL = {"Entity", "Chunk", "KGFControl", "KGFLock", "KGFDocument",
              "KGFEntityVersion", "Query"}

driver = GraphDatabase.driver("bolt://user-konrad.jelen-kgf-neo4j3:7687",
                              auth=("neo4j", "kgfoundry"))
with driver.session() as s:
    docs = s.run(
        "MATCH (d:KGFDocument) RETURN d.id AS id, d.name AS name "
        "ORDER BY d.created_at"
    ).data()
    ents = s.run(
        "MATCH (e:Entity) "
        "RETURN [l IN labels(e) WHERE NOT l IN $struct] AS types, "
        "e.source_documents AS sources",
        struct=list(STRUCTURAL),
    ).data()
driver.close()

# per-doc observation lists: an entity contributes its types to every doc it appears in
per_doc: dict[str, list[str]] = {d["id"]: [] for d in docs}
for e in ents:
    for src in e["sources"] or []:
        if src in per_doc:
            per_doc[src].extend(e["types"])

stream = [(d["name"], per_doc[d["id"]]) for d in docs]
n_obs = sum(len(o) for _, o in stream)
print(f"replay stream: {len(stream)} documents, {n_obs} type observations, "
      f"{len(set(t for _, o in stream for t in o))} distinct types")
print("known bias: stored types are post-remap (the realized stream, not the "
      "counterfactual unremapped one)")

2026-07-06 17:14:00.183 | INFO     | knowledge_graph_foundry.config:<module>:40 - PROJ_ROOT path is: /home/lab/workspace/learning/projects/knowledge-graph-foundry
replay stream: 208 documents, 2086 type observations, 22 distinct types
known bias: stored types are post-remap (the realized stream, not the counterfactual unremapped one)


In [2]:
# candidate gates - each replays the stream through the REAL production classes
# (StabilityMetrics feeds CuringDetector exactly as buffer.py does in the pipeline)

def replay(stream, cfg, floor_obs=0, point_estimate=False):
    """Return the 1-based doc index where the gate cures, or None.
    floor_obs/point_estimate emulate candidate (c): point-estimate missing
    mass n1/N <= threshold behind a hardcoded observation floor."""
    metrics = StabilityMetrics()
    detector = CuringDetector(cfg)
    census: Counter = Counter()
    for i, (_, obs) in enumerate(stream, 1):
        census.update(obs)
        rec = metrics.record(dict(census))
        detector.record(rec)
        if point_estimate:
            total = rec.get("total_occurrences", 0.0)
            n1 = rec.get("singletons", 0.0)
            ok_evidence = total >= floor_obs and total and (n1 / total) <= cfg.missing_mass_threshold
            # candidate (c): same converged/plateau logic, evidence = point+floor
            detector.has_sufficient_evidence = lambda ok=ok_evidence: ok
        cure, reason = detector.should_cure()
        if cure:
            return i, reason
    return None, "never"


def gate_v1(stream):
    # candidate (a): pre-DEF-3 semantics - no evidence gate (threshold 1e9 disables it)
    cfg = CuringSettings(missing_mass_threshold=1e9, max_fluid_documents=10**9)
    return replay(stream, cfg)


def gate_chao1(stream):
    # candidate (b): composite convergence only (plateau disabled via impossible window)
    cfg = CuringSettings(missing_mass_threshold=1e9, max_fluid_documents=10**9)
    metrics = StabilityMetrics()
    detector = CuringDetector(cfg)
    census: Counter = Counter()
    for i, (_, obs) in enumerate(stream, 1):
        census.update(obs)
        detector.record(metrics.record(dict(census)))
        if detector.is_converged():
            return i, "converged"
    return None, "never"


def gate_point_floor(stream, floor=200):
    # candidate (c): the rejected first patch
    cfg = CuringSettings(max_fluid_documents=10**9)
    return replay(stream, cfg, floor_obs=floor, point_estimate=True)


def gate_ucb(stream):
    # candidate (d): shipped DEF-3 fix - missing-mass UCB, no count anywhere
    cfg = CuringSettings(max_fluid_documents=10**9)
    return replay(stream, cfg)


print("gates defined")

gates defined


In [3]:
# retrospective saturation band + run all candidates at three truncations
def saturation_doc(stream, frac=0.95):
    """First doc where the cumulative type inventory reaches frac of its final size."""
    seen, curve = set(), []
    for _, obs in stream:
        seen.update(obs)
        curve.append(len(seen))
    final = curve[-1]
    for i, c in enumerate(curve, 1):
        if c >= frac * final:
            return i, final
    return len(stream), final


results = {}
for cut in (100, 200, len(stream)):
    sub = stream[:cut]
    sat, final_types = saturation_doc(sub)
    results[cut] = {
        "saturation_doc_95pct": sat,
        "final_types": final_types,
        "a_v1_plateau": gate_v1(sub),
        "b_chao1_composite": gate_chao1(sub),
        "c_point_plus_200floor": gate_point_floor(sub),
        "d_missing_mass_ucb": gate_ucb(sub),
    }

for cut, r in results.items():
    print(f"\n=== stream truncated at {cut} docs "
          f"(types={r['final_types']}, 95% saturation at doc {r['saturation_doc_95pct']}) ===")
    for k in ("a_v1_plateau", "b_chao1_composite", "c_point_plus_200floor", "d_missing_mass_ucb"):
        doc, reason = r[k]
        print(f"  {k:<24} cures at doc {doc} ({reason})")


=== stream truncated at 100 docs (types=17, 95% saturation at doc 96) ===
  a_v1_plateau             cures at doc 5 (plateau)
  b_chao1_composite        cures at doc 10 (converged)
  c_point_plus_200floor    cures at doc 26 (plateau)
  d_missing_mass_ucb       cures at doc 20 (plateau)

=== stream truncated at 200 docs (types=21, 95% saturation at doc 138) ===
  a_v1_plateau             cures at doc 5 (plateau)
  b_chao1_composite        cures at doc 10 (converged)
  c_point_plus_200floor    cures at doc 26 (plateau)
  d_missing_mass_ucb       cures at doc 20 (plateau)

=== stream truncated at 208 docs (types=22, 95% saturation at doc 168) ===
  a_v1_plateau             cures at doc 5 (plateau)
  b_chao1_composite        cures at doc 10 (converged)
  c_point_plus_200floor    cures at doc 26 (plateau)
  d_missing_mass_ucb       cures at doc 20 (plateau)


In [4]:
# verdict against the pre-registered acceptance bar + persist
import datetime

full = results[len(stream)]
sat = full["saturation_doc_95pct"]
band = (sat, len(stream))  # inside the band = at or after 95% saturation

verdict_rows = []
for k in ("a_v1_plateau", "b_chao1_composite", "c_point_plus_200floor", "d_missing_mass_ucb"):
    doc, reason = full[k]
    inside = doc is not None and band[0] <= doc <= band[1]
    cure_docs = [results[c][k][0] for c in results]
    verdict_rows.append({"gate": k, "cure_doc": doc, "reason": reason,
                         "inside_saturation_band": inside,
                         "cure_doc_at_truncations": cure_docs})
    print(f"{k:<24} doc={doc} inside_band={inside} truncations={cure_docs}")

out = Path("../reports") / (
    "curing-gate-h32-" + datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d-%H%M%S") + ".json"
)
out.write_text(json.dumps({"band": band, "results": {str(k): v for k, v in results.items()},
                           "verdicts": verdict_rows}, indent=2, default=str))
print("saved", out)

a_v1_plateau             doc=5 inside_band=False truncations=[5, 5, 5]
b_chao1_composite        doc=10 inside_band=False truncations=[10, 10, 10]
c_point_plus_200floor    doc=26 inside_band=False truncations=[26, 26, 26]
d_missing_mass_ucb       doc=20 inside_band=False truncations=[20, 20, 20]
saved ../reports/curing-gate-h32-20260706-151400.json


In [5]:
# ground-truth interrogation: type-inventory saturation vs forward observation coverage
# A late rare type moves the inventory band but may carry negligible mass. For an
# ontology the operative question is: at cure time, what fraction of FUTURE
# observations already have a known type?

def forward_coverage(stream, cure_doc):
    known = set()
    for _, obs in stream[:cure_doc]:
        known.update(obs)
    future = [t for _, obs in stream[cure_doc:] for t in obs]
    if not future:
        return None
    return sum(1 for t in future if t in known) / len(future)


# new-type arrival curve with the observation mass each late type carries
first_seen = {}
mass = Counter()
for i, (_, obs) in enumerate(stream, 1):
    for t in obs:
        mass[t] += 1
        first_seen.setdefault(t, i)

total_mass = sum(mass.values())
print("type arrivals (doc of first appearance, share of all observations):")
for t, doc in sorted(first_seen.items(), key=lambda kv: kv[1]):
    print(f"  doc {doc:>3}  {mass[t]/total_mass:6.2%}  {t}")

print("\nforward observation coverage at each gate's cure point:")
for k in ("a_v1_plateau", "b_chao1_composite", "c_point_plus_200floor", "d_missing_mass_ucb"):
    doc, _ = full[k]
    cov = forward_coverage(stream, doc)
    print(f"  {k:<24} cure@{doc:>3} -> future obs covered: {cov:.2%}")

type arrivals (doc of first appearance, share of all observations):
  doc   1  12.51%  Device
  doc   1  10.88%  Therapy
  doc   1   2.88%  OperatingMode
  doc   1   2.40%  SoftwareTool
  doc   1  14.00%  Index
  doc   1  20.95%  Measurement
  doc   1  21.72%  Parameter
  doc  13   3.60%  Condition
  doc  13   3.16%  Event
  doc  13   3.07%  ApneaEvent
  doc  13   2.92%  Concept
  doc  14   0.14%  PhysiologicalChange
  doc  15   0.14%  Disorder
  doc  15   0.29%  PhysiologicalParameter
  doc  82   0.14%  Procedure
  doc  91   0.05%  PhysiologicalEvent
  doc  96   0.77%  Factor
  doc 136   0.10%  Symptom
  doc 138   0.05%  Threshold
  doc 138   0.05%  Study
  doc 168   0.14%  SeverityLevel
  doc 203   0.05%  Outcome

forward observation coverage at each gate's cure point:
  a_v1_plateau             cure@  5 -> future obs covered: 85.14%
  b_chao1_composite        cure@ 10 -> future obs covered: 84.91%
  c_point_plus_200floor    cure@ 26 -> future obs covered: 98.51%
  d_missing_mass_ucb